In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.ensemble import RandomForestRegressor

In [ ]:
df=pd.read_csv("../data/processed/daily_sales_processed.csv")
df.head()

In [11]:
X=df[['Year','Month','Day','Weekday','Lag_1','Lag_7','Lag_30','Rolling_7_mean','Rolling_30_mean']]
y=df['Sales']

In [13]:
train_size=int(len(df)*0.8)
X_train=X[:train_size]
y_train=y[:train_size]
X_test=X[train_size:]
y_test=y[train_size:]

In [16]:
best_rf=RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_split=2,
    random_state=42
)
best_rf.fit(X_train,y_train)
y_pred=best_rf.predict(X_test)

In [ ]:
#Actual vs Predicted graph
plt.figure(figsize=(12,5))

plt.plot(y_test.values, label='Actual')
plt.plot(y_pred, label='Predicted')

plt.title("Actual vs Predicted Sales")
plt.xlabel("Days")
plt.ylabel("Sales")

plt.legend()
plt.show()

#From the graph we conclude,
'''The Random Forest model successfully captured the overall sales trend and seasonality but struggled to predict extreme sales spikes. 
This suggests that additional business-related features such as promotions, discounts, holidays, or special events may be required to improve forecasting accuracy.'''

In [ ]:
#Residual Analysis
residuals = y_test - y_pred

plt.figure(figsize=(10,5))
plt.hist(residuals, bins=30)

plt.title("Residual Distribution")
plt.xlabel("Error")
plt.ylabel("Frequency")

plt.show()
'''The residual distribution is centered close to zero, indicating that the model does not exhibit significant systematic bias. 
However, the presence of a long positive tail suggests that the model frequently underestimates extreme sales spikes. 
This indicates that additional explanatory variables such as promotions, holidays, or special events may be required to accurately capture high-sales periods.'''

In [ ]:
#Actual vs Predicted Scatter Plot
plt.figure(figsize=(8,6))

plt.scatter(y_test, y_pred)

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")

plt.title("Actual vs Predicted Sales")

plt.show()

'''The scatter plot shows a positive relationship between actual and predicted sales, indicating that the model captures the overall sales behavior. 
However, substantial dispersion around the ideal diagonal line suggests prediction errors remain significant. 
The model particularly struggles with extreme sales values, tending to underestimate large spikes and regress predictions toward average sales levels.'''

In [ ]:
#Feature Importance Visualization
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

plt.figure(figsize=(10,5))

plt.bar(
    importance['Feature'],
    importance['Importance']
)

plt.title("Feature Importance")
plt.xlabel("Features")
plt.ylabel("Importance")

plt.xticks(rotation=45)

plt.show()

In [ ]:
#Final Results Table
results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'RF + Feature Engineering'
    ],
    'MAE': [
        1767.67,
        1724.02,
        1690.40
    ],
    'RMSE': [
        2392.96,
        2425.09,
        2279.05
    ],
    'R2': [
        0.061,
        0.036,
        0.163
    ]
})

results

In [ ]:
#Summary
'''Best Model: RandomForestRegressor with following hyperparamters:
max_depth': 5 
min_samples_split: 2 
n_estimators': 200
'''
# Conclusions

'''1. Feature engineering significantly improved model performance.

2. Lag and rolling window features were the most important predictors.

3. Random Forest outperformed Linear Regression after feature engineering.

4. The model successfully captured overall sales trends.

5. The model struggled to predict extreme sales spikes.

6. Additional business features such as promotions,
   discounts, holidays and events may improve performance.'''